# Homework

In this homework, we'll practice streaming with Kafka (Redpanda) and PyFlink.

We use Redpanda, a drop-in replacement for Kafka. It implements the same
protocol, so any Kafka client library works with it unchanged.

For this homework we will be using Green Taxi Trip data from October 2025:

- [green_tripdata_2025-10.parquet](https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet)

## Setup

We'll use the same infrastructure from the [workshop](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/07-streaming/workshop).

Follow the setup instructions: build the Docker image, start the services:

```bash
cd 07-streaming/workshop/
docker compose build
docker compose up -d
```

This gives us:

- Redpanda (Kafka-compatible broker) on `localhost:9092`
- Flink Job Manager at http://localhost:8081
- Flink Task Manager
- PostgreSQL on `localhost:5432` (user: `postgres`, password: `postgres`)

If you previously ran the workshop and have old containers/volumes,
do a clean start:

```bash
docker compose down -v
docker compose build
docker compose up -d
```

Note: the container names (like `workshop-redpanda-1`) assume the
directory is called `workshop`. If you renamed it, adjust accordingly.


## Question 1. Redpanda version

Run `rpk version` inside the Redpanda container:

```bash
docker exec -it homework7-redpanda-1 rpk version
```

What version of Redpanda are you running?

In [1]:
!docker exec -it homework7-redpanda-1 rpk version

rpk version: v25.3.9
Git ref:     836b4a36ef6d5121edbb1e68f0f673c2a8a244e2
Build date:  2026 Feb 26 07 48 21 Thu
OS/Arch:     linux/amd64
Go version:  go1.24.3

Redpanda Cluster
  node-1  v25.3.9 - 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2


## Question 2. Sending data to Redpanda

Create a topic called `green-trips`:

```bash
docker exec -it workshop-redpanda-1 rpk topic create green-trips
```

Now write a producer to send the green taxi data to this topic.

Read the parquet file and keep only these columns:

- `lpep_pickup_datetime`
- `lpep_dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `passenger_count`
- `trip_distance`
- `tip_amount`
- `total_amount`

Convert each row to a dictionary and send it to the `green-trips` topic.
You'll need to handle the datetime columns - convert them to strings
before serializing to JSON.

Measure the time it takes to send the entire dataset and flush:

```python
from time import time

t0 = time()

# send all rows ...

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')
```

How long did it take to send the data?

In [2]:
import pandas as pd
columns = ["lpep_pickup_datetime",
           "lpep_dropoff_datetime",
           "PULocationID",
           "DOLocationID",
           "passenger_count",
           "trip_distance",
           "tip_amount",
           "total_amount"]
df = pd.read_parquet("green_tripdata_2025-10.parquet")
df = df[columns]

In [3]:
import dataclasses
import json

@dataclasses.dataclass
class Ride:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: float
    trip_distance: float
    tip_amount: float
    total_amount: float

In [4]:
def ride_from_row(row: pd.Series) -> Ride:
    return Ride(
        lpep_pickup_datetime=row["lpep_pickup_datetime"].isoformat(),
        lpep_dropoff_datetime=row["lpep_dropoff_datetime"].isoformat(),
        PULocationID=int(row["PULocationID"]),
        DOLocationID=int(row["DOLocationID"]),
        passenger_count=float(row["passenger_count"]),
        trip_distance=float(row["trip_distance"]),
        tip_amount=float(row["tip_amount"]),
        total_amount=float(row["total_amount"])
    )
    
def ride_serializer(ride: Ride) -> bytes:
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

def ride_deserializer(data: bytes) -> Ride:
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [5]:
from time import time
from kafka import KafkaProducer

t0 = time()
server = 'localhost:9092'
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

topic_name = 'green-trips'

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 17.88 seconds


## Question 3. Consumer - trip distance

Write a Kafka consumer that reads all messages from the `green-trips` topic
(set `auto_offset_reset='earliest'`).

Count how many trips have a `trip_distance` greater than 5.0 kilometers.

How many trips have `trip_distance` > 5?

In [6]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    consumer_timeout_ms=10000,
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

count = 0
total = 0

for message in consumer:
    trip = message.value
    total += 1
    if trip['trip_distance'] > 5.0:
        count += 1

print(f'Trips with trip_distance > 5: {count}')

Trips with trip_distance > 5: 8506


## Part 2: PyFlink (Questions 4-6)

For the PyFlink questions, you'll adapt the workshop code to work with
the green taxi data. The key differences from the workshop:

- Topic name: `green-trips` (instead of `rides`)
- Datetime columns use `lpep_` prefix (instead of `tpep_`)
- You'll need to handle timestamps as strings (not epoch milliseconds)

You can convert string timestamps to Flink timestamps in your source DDL:

```sql
lpep_pickup_datetime VARCHAR,
event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
```

Before running the Flink jobs, create the necessary PostgreSQL tables
for your results.

Important notes for the Flink jobs:

- Place your job files in `workshop/src/job/` - this directory is
  mounted into the Flink containers at `/opt/src/job/`
- Submit jobs with:
  `docker exec -it homework7-jobmanager-1 flink run -py /opt/src/job/your_job.py`
- The `green-trips` topic has 1 partition, so set parallelism to 1
  in your Flink jobs (`env.set_parallelism(1)`). With higher parallelism,
  idle consumer subtasks prevent the watermark from advancing.
- Flink streaming jobs run continuously. Let the job run for a minute
  or two until results appear in PostgreSQL, then query the results.
  You can cancel the job from the Flink UI at http://localhost:8081
- If you sent data to the topic multiple times, delete and recreate
  the topic to avoid duplicates:
  `docker exec -it homework7-redpanda-1 rpk topic delete green-trips`


## Question 4. Tumbling window - pickup location

Create a Flink job that reads from `green-trips` and uses a 5-minute
tumbling window to count trips per `PULocationID`.

Write the results to a PostgreSQL table with columns:
`window_start`, `PULocationID`, `num_trips`.

After the job processes all data, query the results:

```sql
SELECT PULocationID, num_trips
FROM <your_table>
ORDER BY num_trips DESC
LIMIT 3;
```

Which `PULocationID` had the most trips in a single 5-minute window?

In [7]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("""
    DROP TABLE IF EXISTS green_trips_tumbling_5min;
    CREATE TABLE green_trips_tumbling_5min (
        window_start TIMESTAMP,
        PULocationID INT,
        num_trips INT,
        PRIMARY KEY (window_start, PULocationID)
    );
""")

```bash
docker exec -it homework7-jobmanager-1 flink run --detached -py /opt/flink/usrlib/tumbling_5min_job.py
```

In [8]:
cur.execute("""
    SELECT PULocationID, num_trips
    FROM green_trips_tumbling_5min
    ORDER BY num_trips DESC
    LIMIT 3;
""")
for row in cur.fetchall():
    print(f"PULocationID: {row[0]}, num_trips: {row[1]}")

PULocationID: 74, num_trips: 15
PULocationID: 74, num_trips: 14
PULocationID: 74, num_trips: 13


## Question 5. Session window - longest streak

Create another Flink job that uses a session window with a 5-minute gap
on `PULocationID`, using `lpep_pickup_datetime` as the event time
with a 5-second watermark tolerance.

A session window groups events that arrive within 5 minutes of each other.
When there's a gap of more than 5 minutes, the window closes.

Write the results to a PostgreSQL table and find the `PULocationID`
with the longest session (most trips in a single session).

How many trips were in the longest session?

In [9]:
cur.execute("""
    DROP TABLE IF EXISTS green_trips_session_5min;
    CREATE TABLE green_trips_session_5min (
        session_start TIMESTAMP,
        session_end TIMESTAMP,
        PULocationID INT,
        num_trips BIGINT,
        PRIMARY KEY (session_start, PULocationID)
    );
""")

```bash
docker exec -it homework7-jobmanager-1 flink run --detached -py /opt/flink/usrlib/session_5min_job.py
```

In [10]:
cur.execute("""
    SELECT PULocationID, num_trips, session_start, session_end
    FROM green_trips_session_5min
    ORDER BY num_trips DESC
    LIMIT 3;
""")
for row in cur.fetchall():
    print(f"PULocationID: {row[0]}, num_trips: {row[1]}, session_start: {row[2]}, session_end: {row[3]}")

## Question 6. Tumbling window - largest tip

Create a Flink job that uses a 1-hour tumbling window to compute the
total `tip_amount` per hour (across all locations).

Which hour had the highest total tip amount?

In [11]:
cur.execute("""
    DROP TABLE IF EXISTS green_trips_tumbling_1h_tip;
    CREATE TABLE green_trips_tumbling_1h_tip (
        window_start TIMESTAMP,
        window_end TIMESTAMP,
        total_tip NUMERIC,
        PRIMARY KEY (window_start)
    );
""")

```bash
docker exec -it homework7-jobmanager-1 flink run --detached -py /opt/flink/usrlib/tumbling_1h_tip_job.py
```

In [13]:
cur.execute("""
    SELECT window_start, window_end, total_tip
    FROM green_trips_tumbling_1h_tip
    ORDER BY total_tip DESC
    LIMIT 3;
""")
for row in cur.fetchall():
    print(f"window_start: {row[0]}, window_end: {row[1]}, total_tip: {row[2]}")

window_start: 2025-10-16 18:00:00, window_end: 2025-10-16 19:00:00, total_tip: 510.86
window_start: 2025-10-09 18:00:00, window_end: 2025-10-09 19:00:00, total_tip: 472.01
window_start: 2025-10-10 17:00:00, window_end: 2025-10-10 18:00:00, total_tip: 470.08


## Submitting the solutions

- Form for submitting: https://courses.datatalks.club/de-zoomcamp-2026/homework/hw7